# Transcrição de imagens
Execute no Google Colab. Use uma pasta pública do Drive e uma chave do OpenRouter.


In [ ]:
%pip -q install requests "gdown>=6,<7" matplotlib pillow


## 1. Chave de acesso
Use o segredo `API_KEY` do Colab ou informe a chave ao executar.


In [ ]:
import base64
import json
import mimetypes
import re
import tempfile
from getpass import getpass
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import requests
from google.colab import files, userdata
from PIL import Image

PASTA_SAIDA = Path("/content/respostas_json")

try:
    API_KEY = userdata.get("API_KEY")
except userdata.SecretNotFoundError:
    API_KEY = None
if not API_KEY:
    API_KEY = getpass("Chave do OpenRouter: ").strip()
if not API_KEY:
    raise ValueError("Informe sua chave do OpenRouter.")


## 2. Imagens
Informe o link da pasta com as imagens.


In [ ]:
LINK_PASTA_IMAGENS = "https://drive.google.com/drive/folders/1WBkMyRY5T4_CVbka497tA0OCVsewcnX5?usp=sharing" #Imagens Recortadas
# LINK_PASTA_IMAGENS = "https://drive.google.com/drive/folders/1DqNaK7l7Hdu4gd87a5gqicgj7IHJ6ao-?usp=sharing" #Imagens Binarizadas


## 3. Modelo
Defina o modelo e seus parâmetros. Use `None` para omitir o esforço.


In [ ]:
CONFIG_MODELO = {
    "modelo": "google/gemini-3.1-pro-preview",
    "esforco_raciocinio": "high",
    "max_tokens": 65536,
}

# CONFIG_MODELO = {
#     "modelo": "google/gemini-3.8-flash",
#     "esforco_raciocinio": None,
#     "max_tokens": 65536,
# }

# CONFIG_MODELO = {
#     "modelo": "openai/gpt-5.6-terra",
#     "esforco_raciocinio": "high",
#     "max_tokens": 128000,
#     "parametro_limite_tokens": "max_completion_tokens",
# }

# CONFIG_MODELO = {
#     "modelo": "anthropic/claude-sonnet-5.5",
#     "esforco_raciocinio": "high",
#     "max_tokens": 128000,
# }



## 4. Baixar e visualizar


In [ ]:
pasta_temporaria = Path(tempfile.mkdtemp())
gdown.download_folder(LINK_PASTA_IMAGENS, output=f"{pasta_temporaria}/", use_cookies=False)
PASTA_IMAGENS = next(pasta_temporaria.iterdir())
NOME_PASTA_DRIVE = PASTA_IMAGENS.name
imagens = [p for p in PASTA_IMAGENS.iterdir()
           if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
imagens.sort(key=lambda p: [int(t) if t.isdigit() else t.casefold() for t in re.split(r"(\d+)", p.name)])
if not imagens:
    raise FileNotFoundError("Nenhuma imagem encontrada na pasta.")

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVO_RESULTADOS = PASTA_SAIDA / f"{NOME_PASTA_DRIVE}-{CONFIG_MODELO['modelo'].split('/')[-1]}.json"
if ARQUIVO_RESULTADOS.exists():
    raise FileExistsError(f"O JSON já existe: {ARQUIVO_RESULTADOS}")
print(f"Imagens: {len(imagens)}")
print(f"Saída: {ARQUIVO_RESULTADOS}")

linhas = (len(imagens) + 4) // 5
fig, eixos = plt.subplots(linhas, 5, figsize=(10, 2 * linhas), squeeze=False)
for eixo, imagem in zip(eixos.flat, imagens):
    with Image.open(imagem) as miniatura:
        miniatura.thumbnail((200, 160))
        eixo.imshow(miniatura)
    eixo.set_title(imagem.name, fontsize=8, wrap=True)
for eixo in eixos.flat:
    eixo.axis("off")
plt.tight_layout()
plt.show()


## 5. Transcrever
O progresso é salvo após cada imagem.


In [ ]:
PROMPT = (
    "Transcreva todo o texto presente nesta imagem, preservando a grafia original. "
    "Marque trechos ilegíveis como [ilegível]. Não utilize quebras de linha. "
    "Retorne apenas a transcrição, sem introdução, explicações ou avaliação."
)
payload = {"model": CONFIG_MODELO["modelo"], "max_tokens": CONFIG_MODELO["max_tokens"]}
if CONFIG_MODELO.get("esforco_raciocinio") is not None:
    payload["reasoning"] = {"effort": CONFIG_MODELO["esforco_raciocinio"]}

resultados = {
    **CONFIG_MODELO,
    "entrada": {"pasta": NOME_PASTA_DRIVE, "link_drive": LINK_PASTA_IMAGENS,
                "quantidade_imagens": len(imagens)},
    "status": "em_andamento",
    "imagens": [],
}
try:
    for indice, imagem in enumerate(imagens, 1):
        print(f"[{indice}/{len(imagens)}] {imagem.name}")
        mime = mimetypes.guess_type(imagem.name)[0]
        base64_imagem = base64.b64encode(imagem.read_bytes()).decode("ascii")
        payload["messages"] = [{"role": "user", "content": [
            {"type": "text", "text": PROMPT},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{base64_imagem}"}},
        ]}]
        resposta = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}"},
            json=payload, timeout=(30, 300),
        )
        resposta.raise_for_status()
        dados = resposta.json()
        if "error" in dados:
            raise RuntimeError(dados["error"])
        escolha = dados["choices"][0]
        if escolha.get("finish_reason") == "length":
            raise RuntimeError("Resposta truncada. Aumente max_tokens na CONFIG_MODELO.")
        texto = escolha["message"].get("content")
        if not isinstance(texto, str) or not texto.strip():
            raise RuntimeError("O modelo não retornou uma transcrição em texto.")
        resultados["imagens"].append({
            "arquivo": imagem.name, "caminho_relativo": imagem.name,
            "texto_transcrito": texto, "custo": dados.get("usage", {}).get("cost"),
        })
        ARQUIVO_RESULTADOS.write_text(json.dumps(resultados, ensure_ascii=False, indent=2), encoding="utf-8")
    resultados["status"] = "concluido"
except Exception as erro:
    resultados["erro"] = str(erro)
    print(f"Geração interrompida: {erro}")
finally:
    ARQUIVO_RESULTADOS.write_text(json.dumps(resultados, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"JSON salvo: {ARQUIVO_RESULTADOS}")


## 6. Baixar o JSON


In [ ]:
files.download(str(ARQUIVO_RESULTADOS))
